# LFW + SurvFace 공통 결과 집계와 시각화

데이터셋별 06 runbook이 만든 fallback-free Step 1 결과만 동일 schema로 합칩니다. 이 노트북은 압축기를 fit하거나 검색을 다시 실행하지 않으며, PCA와 PQ를 서로 독립된 family로 비교합니다.

성공 기준:

- LFW와 SurvFace의 `result_manifest.json` 및 CSV hash가 일치합니다.
- `paired_embedding_metrics`와 `compare_cosine_retrieval`의 필수 열을 모두 포함합니다.
- `origin_fallback_used`가 한 행이라도 True이면 즉시 중단합니다.
- PCA/PQ 결합 profile 또는 legacy fallback artifact를 받아들이지 않습니다.
- 동일 scope/model의 요약표와 저장량-오차/성능 그림을 생성합니다.


In [ ]:
from __future__ import annotations

# 집계 범위: 데이터셋 06 노트북과 동일하게 설정
MODE = "dev"              # "dev" 또는 "real"
DATA_FRACTION = 1.0      # 0 < DATA_FRACTION <= 1
SEED = 42

MODEL_NAME = "arcface"
DATASETS = ("lfw", "survface")
RUN_IDS = {
    "lfw": None,          # 여러 후보가 있으면 명시적 extraction run_id 입력
    "survface": None,
}
EXECUTE_STAGE = True
WRITE_OUTPUTS = True
OVERWRITE = True          # canonical stage result replacement

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("D:/ronbun 내부에서 노트북을 실행하십시오.")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.evaluation import (
    PAIRED_EMBEDDING_COLUMNS,
    RETRIEVAL_COMPARISON_COLUMNS,
    compare_cosine_retrieval,
    paired_embedding_metrics,
)
from research.experiments.scope import ExperimentScope
from research.runtime.hashing import sha256_file

EXPERIMENT_SCOPE = ExperimentScope(
    mode=MODE, data_fraction=DATA_FRACTION, seed=SEED
)
RESULT_ROOT = PROJECT_ROOT / "results" / "step1"
OUTPUT_ROOT = RESULT_ROOT / "common"
SCOPE_TAG = f"{MODE}_p{DATA_FRACTION:.4f}_s{SEED}_{MODEL_NAME}"

display(pd.Series({
    **EXPERIMENT_SCOPE.as_dict(),
    "model": MODEL_NAME,
    "datasets": DATASETS,
    "execute_stage": EXECUTE_STAGE,
    "write_outputs": WRITE_OUTPUTS,
    "scope_tag": SCOPE_TAG,
    "paired_api": paired_embedding_metrics.__name__,
    "retrieval_api": compare_cosine_retrieval.__name__,
}, name="value").to_frame())


## Plan

1. Dataset별 결과 manifest를 scope/model/run ID로 정확히 선택합니다.
2. 각 CSV의 SHA-256, 필수 열, 독립 compression family, fallback 불사용을 검사합니다.
3. Dataset 열을 붙여 paired/retrieval/summary 표를 결합합니다.
4. 저장 bytes 대비 angular error와 DIR/FPIR를 공통 축으로 시각화합니다.
5. 필요할 때만 별도 common 결과 디렉터리에 표와 그림을 저장합니다.


In [ ]:
def matching_manifests(dataset: str) -> list[Path]:
    dataset_root = RESULT_ROOT / dataset
    if not dataset_root.is_dir():
        return []
    candidates = []
    for path in sorted(dataset_root.glob("*/*/result_manifest.json")):
        payload = json.loads(path.read_text(encoding="utf-8"))
        if payload.get("dataset") != dataset:
            continue
        if payload.get("model_name") != MODEL_NAME:
            continue
        scope = payload.get("scope", {})
        if (
            scope.get("mode") != MODE
            or float(scope.get("data_fraction", -1.0)) != DATA_FRACTION
            or int(scope.get("seed", -1)) != SEED
        ):
            continue
        requested_run = RUN_IDS.get(dataset)
        if requested_run is not None and payload.get("source", {}).get("extraction_run_id") != requested_run:
            continue
        candidates.append(path)
    return candidates

selected_manifests = {}
discovery = {}
for dataset in DATASETS:
    candidates = matching_manifests(dataset)
    discovery[dataset] = [str(path) for path in candidates]
    if len(candidates) == 1:
        selected_manifests[dataset] = candidates[0]
    elif EXECUTE_STAGE:
        if not candidates:
            raise FileNotFoundError(
                f"{dataset}: scope={SCOPE_TAG} 결과 manifest가 없습니다. "
                "먼저 dataset 06 notebook을 실행하십시오."
            )
        raise RuntimeError(
            f"{dataset}: 일치하는 결과가 {len(candidates)}개입니다. RUN_IDS에 extraction run_id를 지정하십시오."
        )

display(pd.Series({
    dataset: len(paths) for dataset, paths in discovery.items()
}, name="matching_manifest_count").to_frame())


## Provenance and schema validation

06 결과의 manifest가 선언한 hash를 다시 계산합니다. API 필수 열 외의 dataset/model/storage/threshold-policy 열은 공통 분석 metadata이며, legacy fallback 열을 성능 지표로 사용하지 않습니다. `origin_decision_threshold`와 `compressed_decision_threshold`를 각각 검증하며, `threshold_crossing`은 각 표현이 정책별 threshold에서 내린 decision의 불일치입니다.


In [ ]:
def verify_file(manifest_path: Path, name: str, metadata: dict) -> Path:
    path = manifest_path.parent / name
    if not path.is_file():
        raise FileNotFoundError(path)
    actual = sha256_file(path)
    if actual != metadata["sha256"]:
        raise ValueError(f"artifact hash mismatch: {path}")
    if path.stat().st_size != int(metadata["bytes"]):
        raise ValueError(f"artifact byte-size mismatch: {path}")
    return path

def require_columns(frame: pd.DataFrame, required, *, label: str) -> None:
    missing = sorted(set(required).difference(frame.columns))
    if missing:
        raise ValueError(f"{label} missing columns: {missing}")

def parse_strict_boolean_series(series: pd.Series, *, label: str) -> pd.Series:
    def parse(value):
        if pd.isna(value):
            raise ValueError(f"{label}: missing boolean value is not allowed.")
        if isinstance(value, (bool, np.bool_)):
            return bool(value)
        if isinstance(value, (int, np.integer)) and int(value) in {0, 1}:
            return bool(int(value))
        if isinstance(value, (float, np.floating)) and float(value) in {0.0, 1.0}:
            return bool(int(value))
        if isinstance(value, str):
            token = value.strip().lower()
            if token in {"true", "1"}:
                return True
            if token in {"false", "0"}:
                return False
        raise ValueError(f"{label}: unknown boolean value {value!r}.")

    return series.map(parse).astype(bool)

def validate_independent_families(frame: pd.DataFrame, *, label: str) -> None:
    family = frame["compression_family"].astype(str).str.lower()
    profile = frame["compression_profile"].astype(str).str.lower()
    if not family.isin({"pca", "pq"}).all():
        raise ValueError(f"{label}: compression_family는 pca/pq만 허용합니다.")
    chained = profile.str.contains("pca") & profile.str.contains("pq")
    if chained.any():
        raise ValueError(f"{label}: PCA->PQ 결합 profile이 포함되어 있습니다.")
    inconsistent = (
        (family.eq("pca") & ~profile.str.startswith("pca_"))
        | (family.eq("pq") & ~profile.str.startswith("pq_512_"))
    )
    if inconsistent.any():
        raise ValueError(f"{label}: family/profile 명칭이 일치하지 않습니다.")

paired_frames = []
retrieval_frames = []
summary_frames = []
loaded_manifests = {}

if EXECUTE_STAGE:
    for dataset, manifest_path in selected_manifests.items():
        payload = json.loads(manifest_path.read_text(encoding="utf-8"))
        if payload.get("compression", {}).get("pca_to_pq") is not False:
            raise ValueError(f"{dataset}: pca_to_pq=false가 아닙니다.")
        if payload.get("evaluation", {}).get("origin_fallback") is not False:
            raise ValueError(f"{dataset}: origin_fallback=false가 아닙니다.")
        declared_columns = payload.get("required_columns", {})
        if declared_columns.get("paired") != list(PAIRED_EMBEDDING_COLUMNS):
            raise ValueError(f"{dataset}: manifest paired schema version mismatch.")
        if declared_columns.get("retrieval") != list(RETRIEVAL_COMPARISON_COLUMNS):
            raise ValueError(f"{dataset}: manifest retrieval schema version mismatch.")

        files = payload.get("files", {})
        expected = {
            "paired_embedding_metrics.csv",
            "retrieval_comparison.csv",
            "compression_summary.csv",
        }
        if set(files) != expected:
            raise ValueError(f"{dataset}: 결과 파일 집합이 예상과 다릅니다: {sorted(files)}")
        paths = {
            name: verify_file(manifest_path, name, files[name])
            for name in sorted(expected)
        }
        paired = pd.read_csv(paths["paired_embedding_metrics.csv"])
        retrieval = pd.read_csv(paths["retrieval_comparison.csv"])
        summary = pd.read_csv(paths["compression_summary.csv"])

        require_columns(paired, PAIRED_EMBEDDING_COLUMNS, label=f"{dataset}/paired")
        require_columns(retrieval, RETRIEVAL_COMPARISON_COLUMNS, label=f"{dataset}/retrieval")
        require_columns(
            summary,
            {
                "compression_family", "compression_profile", "threshold_policy",
                "dir_rank1", "fpir", "agreement_with_origin",
                "threshold_crossing_rate", "mean_angular_error_rad",
                "storage_bytes_per_embedding", "codebook_bytes",
                "codebook_bytes_source",
            },
            label=f"{dataset}/summary",
        )
        validate_independent_families(paired, label=f"{dataset}/paired")
        validate_independent_families(retrieval, label=f"{dataset}/retrieval")
        validate_independent_families(summary, label=f"{dataset}/summary")
        manifest_storage = payload.get("compression", {}).get("profile_storage")
        if not isinstance(manifest_storage, dict):
            raise ValueError(f"{dataset}: manifest profile_storage is missing.")
        summary_profiles = set(summary["compression_profile"].astype(str))
        if set(manifest_storage) != summary_profiles:
            raise ValueError(f"{dataset}: manifest/summary profile_storage mismatch.")
        for profile, rows in summary.groupby("compression_profile", sort=False):
            expected_storage = manifest_storage[str(profile)]
            observed = rows.iloc[0]
            expected_storage_keys = {
                "compression_family", "storage_bytes_per_embedding",
                "codebook_bytes", "codebook_bytes_source",
            }
            if set(expected_storage) != expected_storage_keys:
                raise ValueError(f"{dataset}/{profile}: profile_storage schema mismatch.")
            if str(observed["compression_family"]) != str(expected_storage["compression_family"]):
                raise ValueError(f"{dataset}/{profile}: compression family mismatch.")
            if int(observed["storage_bytes_per_embedding"]) != int(expected_storage["storage_bytes_per_embedding"]):
                raise ValueError(f"{dataset}/{profile}: storage bytes mismatch.")
            if int(observed["codebook_bytes"]) != int(expected_storage["codebook_bytes"]):
                raise ValueError(f"{dataset}/{profile}: codebook bytes mismatch.")
            if str(observed["codebook_bytes_source"]) != str(expected_storage["codebook_bytes_source"]):
                raise ValueError(f"{dataset}/{profile}: codebook source mismatch.")

        if parse_strict_boolean_series(
            paired["origin_fallback_used"], label=f"{dataset}/paired origin_fallback_used"
        ).any():
            raise ValueError(f"{dataset}: paired 결과에 origin fallback 사용 행이 있습니다.")
        if parse_strict_boolean_series(
            retrieval["origin_fallback_used"], label=f"{dataset}/retrieval origin_fallback_used"
        ).any():
            raise ValueError(f"{dataset}: retrieval 결과에 origin fallback 사용 행이 있습니다.")

        paired["dataset"] = dataset
        retrieval["dataset"] = dataset
        summary["dataset"] = dataset
        summary["model_name"] = payload["model_name"]
        paired_frames.append(paired)
        retrieval_frames.append(retrieval)
        summary_frames.append(summary)
        loaded_manifests[dataset] = payload

    paired_all = pd.concat(paired_frames, ignore_index=True)
    retrieval_all = pd.concat(retrieval_frames, ignore_index=True)
    summary_all = pd.concat(summary_frames, ignore_index=True)
    display(summary_all.sort_values(
        ["threshold_policy", "dataset", "compression_family", "storage_bytes_per_embedding"]
    ))
else:
    paired_all = None
    retrieval_all = None
    summary_all = None
    print("EXECUTE_STAGE=False: 결과 CSV를 읽지 않았습니다.")


## Common visualizations

PCA의 저장 bytes는 FP32 좌표 payload이고 PQ는 code payload입니다. PQ codebook/scale metadata는 별도 overhead이므로 이 그림만으로 총 저장량 절감을 단정하지 않습니다.


In [ ]:
figures = {}
if EXECUTE_STAGE:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for (dataset, family), group in summary_all.groupby(
        ["dataset", "compression_family"], sort=True
    ):
        distortion = (
            group.sort_values("storage_bytes_per_embedding")
            .drop_duplicates(["compression_profile"])
        )
        axes[0].plot(
            distortion["storage_bytes_per_embedding"],
            distortion["mean_angular_error_rad"],
            marker="o",
            label=f"{dataset}/{family}",
        )
    axes[0].set(
        xlabel="Representation payload (bytes / embedding)",
        ylabel="Mean angular error (rad)",
        title="Storage vs. embedding distortion",
    )
    axes[0].grid(alpha=0.25)
    axes[0].legend()

    frozen = summary_all.loc[
        summary_all["threshold_policy"].eq("frozen_origin")
    ]
    for (dataset, family), group in frozen.groupby(
        ["dataset", "compression_family"], sort=True
    ):
        ordered = group.sort_values("storage_bytes_per_embedding")
        axes[1].plot(
            ordered["storage_bytes_per_embedding"],
            ordered["dir_rank1"],
            marker="o",
            label=f"{dataset}/{family}",
        )
    axes[1].set(
        xlabel="Representation payload (bytes / embedding)",
        ylabel="DIR rank-1",
        title=f"Frozen-threshold DIR (target FPIR differs by dataset)",
    )
    axes[1].grid(alpha=0.25)
    axes[1].legend()
    fig.tight_layout()
    figures["storage_distortion_and_dir"] = fig

    fpir_table = summary_all.pivot_table(
        index=["dataset", "compression_family", "compression_profile"],
        columns="threshold_policy",
        values=["dir_rank1", "fpir", "threshold_crossing_rate"],
    ).sort_index()
    display(fpir_table)
else:
    print("EXECUTE_STAGE=False: 그림을 생성하지 않았습니다.")


## Export common tables and figures

공통 산출물은 dataset 06 원본을 수정하지 않고 별도 디렉터리에 기록합니다. 같은 scope 경로가 이미 있으면 덮어쓰지 않습니다.


In [ ]:
export_result = {"status": "not_executed"}
if EXECUTE_STAGE:
    destination = OUTPUT_ROOT / SCOPE_TAG
    export_result = {
        "status": "computed_not_written",
        "destination": str(destination),
        "datasets": list(DATASETS),
        "summary_rows": int(len(summary_all)),
    }
    if WRITE_OUTPUTS:
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.mkdir(exist_ok=False)
        paired_path = destination / "paired_embedding_metrics_all.csv"
        retrieval_path = destination / "retrieval_comparison_all.csv"
        summary_path = destination / "compression_summary_all.csv"
        figure_path = destination / "storage_distortion_and_dir.png"
        paired_all.to_csv(paired_path, index=False, encoding="utf-8", lineterminator="\n")
        retrieval_all.to_csv(retrieval_path, index=False, encoding="utf-8", lineterminator="\n")
        summary_all.to_csv(summary_path, index=False, encoding="utf-8", lineterminator="\n")
        figures["storage_distortion_and_dir"].savefig(
            figure_path, dpi=180, bbox_inches="tight"
        )
        source_manifests = {
            dataset: {
                "path": str(path.relative_to(PROJECT_ROOT)),
                "sha256": sha256_file(path),
            }
            for dataset, path in selected_manifests.items()
        }
        common_manifest = {
            "schema_version": 1,
            "scope": EXPERIMENT_SCOPE.as_dict(),
            "model_name": MODEL_NAME,
            "datasets": list(DATASETS),
            "origin_fallback": False,
            "pca_to_pq": False,
            "profile_storage": {
                dataset: loaded_manifests[dataset]["compression"]["profile_storage"]
                for dataset in DATASETS
            },
            "source_manifests": source_manifests,
            "files": {
                path.name: {"sha256": sha256_file(path), "bytes": path.stat().st_size}
                for path in (paired_path, retrieval_path, summary_path, figure_path)
            },
        }
        manifest_path = destination / "result_manifest.json"
        manifest_path.write_text(
            json.dumps(common_manifest, ensure_ascii=False, indent=2, sort_keys=True) + "\n",
            encoding="utf-8",
        )
        export_result = {
            "status": "written",
            "destination": str(destination),
            "manifest": str(manifest_path),
        }
export_result


## Final check

- Dataset별 source manifest hash와 공통 manifest hash가 일치하는지 확인합니다.
- `real, 1.0`이 아닌 결과는 개발/규모 ablation으로만 표시합니다.
- LFW와 SurvFace는 protocol과 목표 FPIR가 다르므로 절대 DIR 수치를 직접 동일 난이도로 해석하지 않습니다.
- PCA와 PQ의 payload 정의가 다르며 PQ codebook metadata를 별도로 보고합니다.
- AdaFace/MagFace adapter가 구현되기 전에는 이 결과를 ArcFace 외 모델로 일반화하지 않습니다.
